# Modélisation

## Data Import

In [ ]:
import pandas as pd
import numpy as np

# 1. Charger le CSV
df = pd.read_csv('../data/avis_cleaned.csv', sep=';')

df.head()

## Text Cleaning avec Spacy

In [ ]:
import pandas as pd
import spacy

# Chargement du modèle français
# (Nécessite: python -m spacy download fr_core_news_sm)
nlp = spacy.load("fr_core_news_sm")

# Version optimisée avec nlp.pipe
def nettoyer_rapide(textes):
    docs = nlp.pipe([str(t).lower() for t in textes], disable=["ner", "parser"], n_process=-1)
    clean_texts = []
    for doc in docs:
        tokens = [t.lemma_ for t in doc if not t.is_stop and not t.is_punct and len(t.text) > 2]
        clean_texts.append(" ".join(tokens))
    return clean_texts

df['avis_spacy'] = nettoyer_rapide(df['commentaire'].tolist())
# Application sur votre colonne d'avis (ex: 'avis_text')
# Note: Sur 40 000 lignes, cela peut prendre quelques minutes

In [ ]:
df.head()

## Entraînement et Classification de Thèmes avec BERTopic

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Configuration de UMAP (Réduction de dimension)
umap_model = UMAP(n_neighbors=15,
                  n_components=5,
                  min_dist=0.0,
                  metric='cosine',
                  random_state=42)

# Configuration de HDBSCAN (Le moteur de clustering)
# min_cluster_size : taille minimale pour qu'un groupe soit considéré comme un "thème"
hdbscan_model = HDBSCAN(
    min_cluster_size=30, 
    metric='euclidean', 
    cluster_selection_method='eom', 
    prediction_data=True
)

# Initialisation de BERTopic avec les modèles UMAP et HDBSCAN configurés
# Le modèle 'paraphrase-multilingual-MiniLM-L12-v2' est excellent pour le français
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=True,
    nr_topics="auto", # 
    verbose=True
)
# 2. Entraînement sur les avis nettoyés
# On utilise la colonne originale ou nettoyée selon vos tests
topics, probs = topic_model.fit_transform(df['avis_spacy'].tolist())

# 3. Voir les sujets principaux découverts
print(topic_model.get_topic_info())

### Sauvegarde

In [ ]:
# 1. Sauvegarde du modèle BERTopic
topic_model.save("modele_bertopic_full", serialization="safetensors", save_ctfidf=True)

# 2. Sauvegarde des résultats dans ton CSV
df['topic_id'] = topics
df.to_csv('avis_40k_final_avec_topics.csv', index=False, sep=';')

print("Sauvegarde terminée ! Le modèle et les résultats sont prêts pour l'analyse et l'application.")

## Visualisation

### Distance Sémantique des thèmes

In [ ]:
# Affiche l'interactivité des thèmes (distance sémantique)
topic_model.visualize_topics()

### Hierarchie des thèmes

In [ ]:
# 1. Visualisez d'abord la hiérarchie pour comprendre les regroupements naturels
hierarchical_topics = topic_model.hierarchical_topics(df['avis_spacy'].tolist())
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

### Reduction du nombre de thèmes

In [ ]:
# --- MODIFIEZ LE NOMBRE DE THEMES ICI ---
NB_THEMES = 9
# -------------------------------

# --- ÉTAPE DE NETTOYAGE CRITIQUE ---
# On réinitialise les probabilités pour éviter l'IndexError lors de la fusion
topic_model.probabilities_ = None

# 1. Réduction du nombre de thèmes
topic_model.reduce_topics(df['avis_spacy'].tolist(), nr_topics=NB_THEMES)

# --- 2. CALCUL DES PROBABILITÉS ALIGNÉES  ---
# On demande au modèle de recalculer les scores sur la base du nouveau découpage
topics, probs = topic_model.transform(df['avis_spacy'].tolist())
# On met à jour le DataFrame
df['topic_id'] = topics

# Affichage rapide pour vérification
print(f"Nouveau découpage en {NB_THEMES} thèmes terminé.")
print(topic_model.get_topic_info().head(NB_THEMES + 1))

In [ ]:
# 1. Le Barchart des 10 thèmes
# On précise n_words=10 pour voir les 10 mots les plus représentatifs de chaque grand thème
fig_bar = topic_model.visualize_barchart(top_n_topics=10, n_words=5)
fig_bar.show()

# 2. La carte des distances (Intertopic Distance Map)
# Idéal pour voir si vos 10 thèmes sont bien distincts ou s'ils se chevauchent encore
fig_map = topic_model.visualize_topics()
fig_map.show()

In [ ]:
# 1. Créer un dictionnaire de correspondance (Mapping)
map_thèmes = {
     -1: "Flux Standard",          # Avis généraux/mélangés
     0: "Vitesse & Conformité",    # Rapide + Conforme
     1: "Logistique & Véhicule",   # Emballage + Identification pièce
     2: "Satisfaction Client",     # "Bien / Parfait"
     3: "Retards de Livraison",    # "Trop long / Délai"
     4: "Fidélité Produit/Photo",  # Photo vs Réalité
     5: "Efficacité 'Nickel'",     # Avis ultra-positifs courts
     6: "Benchmark Concurrence",   # Références à Amazon/Autres
     7: "Montage & Installation",   # Facilité d'usage
}

# 2. Ajouter les résultats au dataframe original
df['topic_label'] = df['topic_id'].map(map_thèmes).fillna("Autres/Inclassables")

# 3. Calculer la note moyenne par thème
analyse_stats = df.groupby(['topic_id','topic_label'])['note'].agg(['mean', 'count']).sort_values(by='mean')
print(analyse_stats)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# Création du graphique à bulles (ou barres)
ax = sns.barplot(x='mean', y='topic_label', data=analyse_stats, palette='RdYlGn')

plt.axvline(x=4, color='blue', linestyle='--', label='Objectif Qualité (4/5)')
plt.title('Satisfaction Client par Thématique (Note Moyenne)', fontsize=15)
plt.xlabel('Note Moyenne (sur 5)')
plt.ylabel('Thématiques identifiées')
plt.xlim(1, 5)
plt.legend()

plt.show()

### Sauvegarde à Topics Réduits

In [ ]:
# 2. SAUVEGARDE DU MODÈLE RÉDUIT (Très important pour l'app)
# On change de nom pour ne pas confondre avec la version 151 thèmes
topic_model.save("modele_bertopic_reduced", serialization="safetensors", save_ctfidf=True)

# 3. SAUVEGARDE DU DATAFRAME FINAL
# C'est ce fichier qui servira à faire tes graphiques dans Streamlit
df.to_csv('avis_oscaro_9_familles.csv', index=False, sep=';')

print("Fichiers prêts pour la production (9 thèmes) !")